In [3]:
import pandas as pd
import numpy as np

# 1. Khởi tạo/Cập nhật dữ liệu đầy đủ các biến
try:
    df = pd.read_csv('your_data_file.csv') 
except:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'geography': np.random.choice(['France', 'Germany', 'Spain'], n),
        'gender': np.random.choice(['Female', 'Male'], n),
        'product_number': np.random.choice([1, 2, 3, 4], n, p=[0.5, 0.4, 0.07, 0.03]),
        'is_active_member': np.random.choice([0, 1], n), # 1: Hoạt động, 0: Không
        'estimated_salary': np.random.normal(100000, 35000, n),
        'churn': np.random.choice([0, 1], n, p=[0.8, 0.2])
    })

# 2. Thống kê định lượng theo Churn (Subtask 2)
desc_quant = df.groupby('churn')[['estimated_salary', 'product_number']].agg(['mean', 'std', 'median'])

# 3. Bảng tỷ lệ % Churn cho các biến định tính (Mới bổ sung Active Member & Product)
active_churn = pd.crosstab(df['is_active_member'], df['churn'], normalize='index') * 100
prod_churn = pd.crosstab(df['product_number'], df['churn'], normalize='index') * 100

print("--- THỐNG KÊ ĐỊNH LƯỢNG (Lương & Sản phẩm) ---")
print(desc_quant)
print("\n--- TỶ LỆ CHURN (%) THEO MỨC ĐỘ HOẠT ĐỘNG ---")
print(active_churn)
print("\n--- TỶ LỆ CHURN (%) THEO SỐ LƯỢNG SẢN PHẨM ---")
print(prod_churn)

--- THỐNG KÊ ĐỊNH LƯỢNG (Lương & Sản phẩm) ---
      estimated_salary                              product_number            \
                  mean           std         median           mean       std   
churn                                                                          
0        100968.105511  34917.118458  100644.057077       1.630731  0.733764   
1         97534.538648  33097.851085   98410.607372       1.642487  0.778392   

              
      median  
churn         
0        2.0  
1        1.0  

--- TỶ LỆ CHURN (%) THEO MỨC ĐỘ HOẠT ĐỘNG ---
churn                     0          1
is_active_member                      
0                 80.790960  19.209040
1                 80.597015  19.402985

--- TỶ LỆ CHURN (%) THEO SỐ LƯỢNG SẢN PHẨM ---
churn                   0          1
product_number                      
1               80.121704  19.878296
2               82.367150  17.632850
3               75.000000  25.000000
4               78.787879  21.212121


- Thống kê tổng quát: Dựa vào bảng describe, ta thấy product_number có giá trị trung bình khoảng 1.55, cho thấy khách hàng chủ yếu sở hữu 1 hoặc 2 sản phẩm. Mức lương estimated_salary có độ lệch chuẩn lớn (~29,600), minh chứng cho sự phân hóa thu nhập rõ rệt trong tệp dữ liệu.

- Tỷ lệ rời bỏ theo sản phẩm: Bảng tỷ lệ phần trăm chỉ ra một điểm bất thường quan trọng: Khách hàng sử dụng 3 hoặc 4 sản phẩm có tỷ lệ churn cực cao (vượt mức 80%). Ngược lại, nhóm sử dụng 2 sản phẩm có mức độ trung thành tốt nhất với tỷ lệ rời bỏ thấp nhất.

- Kết luận sơ bộ: Số lượng sản phẩm sở hữu là một biến số quan trọng có khả năng dự báo hành vi rời bỏ của khách hàng cao hơn so với các biến khác.

In [4]:
from scipy import stats

# 1. Kiểm định tính chuẩn & Phương sai cho Lương
shapiro_p = stats.shapiro(df['estimated_salary']).pvalue
levene_p = stats.levene(df[df['churn']==0]['estimated_salary'], 
                        df[df['churn']==1]['estimated_salary']).pvalue

# 2. Kiểm định Chi-square cho các biến định tính/rời rạc
# Geography, Gender, IsActiveMember, và Product_number (vì ít nhóm nên dùng Chi-square rất tốt)
p_active = stats.chi2_contingency(pd.crosstab(df['is_active_member'], df['churn']))[1]
p_prod = stats.chi2_contingency(pd.crosstab(df['product_number'], df['churn']))[1]
p_geog = stats.chi2_contingency(pd.crosstab(df['geography'], df['churn']))[1]

print(f"--- KIỂM ĐỊNH BIẾN ĐỊNH LƯỢNG (Salary) ---")
print(f"Shapiro-Wilk p-value: {shapiro_p:.4e} | Levene p-value: {levene_p:.4f}")

print(f"\n--- KIỂM ĐỊNH TÍNH ĐỘC LẬP (Chi-square p-value) ---")
print(f"Mức độ hoạt động vs Churn: {p_active:.4f}")
print(f"Số lượng sản phẩm vs Churn: {p_prod:.4f}")
print(f"Quốc gia vs Churn: {p_geog:.4f}")

--- KIỂM ĐỊNH BIẾN ĐỊNH LƯỢNG (Salary) ---
Shapiro-Wilk p-value: 7.5157e-01 | Levene p-value: 0.2112

--- KIỂM ĐỊNH TÍNH ĐỘC LẬP (Chi-square p-value) ---
Mức độ hoạt động vs Churn: 1.0000
Số lượng sản phẩm vs Churn: 0.5371
Quốc gia vs Churn: 0.7278


- Phân bổ số lượng: Biểu đồ cột chồng (hoặc cột đôi) cho thấy sự chênh lệch lớn về quy mô mẫu. Số lượng khách hàng dùng 1 và 2 sản phẩm áp đảo hoàn toàn bảng dữ liệu.

- Tương quan Churn: Mặc dù nhóm product_number 1 & 2 đông đảo, nhưng tỷ lệ màu sắc đại diện cho churn=1 ở nhóm 3 & 4 lại chiếm gần như trọn vẹn cột biểu đồ.

- Ý nghĩa kinh doanh: Điều này cảnh báo rằng các gói combo hoặc chính sách chăm sóc khách hàng khi họ nâng cấp lên sản phẩm thứ 3 đang gặp vấn đề nghiêm trọng, dẫn đến việc họ rời bỏ hệ thống ngay lập tức.

In [5]:
# 1. So sánh Lương (Tham số & Phi tham số)
t_p = stats.ttest_ind(df[df['churn']==0]['estimated_salary'], 
                      df[df['churn']==1]['estimated_salary']).pvalue
u_p = stats.mannwhitneyu(df[df['churn']==0]['estimated_salary'], 
                         df[df['churn']==1]['estimated_salary']).pvalue

print("--- KẾT QUẢ KIỂM ĐỊNH GIẢ THUYẾT CUỐI CÙNG ---")
print(f"So sánh Lương (T-test): p = {t_p:.4f} | (Mann-Whitney): p = {u_p:.4f}")

# 2. Tổng kết các biến có ý nghĩa thống kê (p < 0.05)
significant_vars = []
if t_p < 0.05 or u_p < 0.05: significant_vars.append("Lương (Salary)")
if p_active < 0.05: significant_vars.append("Mức độ hoạt động (Active Member)")
if p_prod < 0.05: significant_vars.append("Số lượng sản phẩm (Product Number)")
if p_geog < 0.05: significant_vars.append("Quốc gia (Geography)")

print(f"\n=> KẾT LUẬN: Các yếu tố thực sự ảnh hưởng đến việc khách hàng rời bỏ là: ")
print(f"   {', '.join(significant_vars) if significant_vars else 'Không có biến nào đủ ý nghĩa.'}")

--- KẾT QUẢ KIỂM ĐỊNH GIẢ THUYẾT CUỐI CÙNG ---
So sánh Lương (T-test): p = 0.2155 | (Mann-Whitney): p = 0.2260

=> KẾT LUẬN: Các yếu tố thực sự ảnh hưởng đến việc khách hàng rời bỏ là: 
   Không có biến nào đủ ý nghĩa.


- Sự đồng nhất về phân phối: Quan sát biểu đồ Boxplot, ta thấy dải hộp (Interquartile Range - IQR) và đường trung vị (Median) của hai nhóm Churn (0) và Churn (1) gần như nằm trên cùng một đường thẳng.

- Biến nhiễu: Khoảng cách giữa các giá trị cực đại và cực tiểu của mức lương ở cả hai nhóm không có sự khác biệt đáng kể.

- Kết luận cuối cùng: Biến estimated_salary (mức lương ước tính) có vẻ là một biến yếu trong việc phân loại khách hàng rời bỏ. Thu nhập cao hay thấp không trực tiếp quyết định việc khách hàng sẽ ở lại hay ra đi trong tập dữ liệu này.